In [14]:
import numpy as np
from tqdm import tqdm

class SortingEnv():
    def __init__(self, n=5):
        self.n = n
        self.goal = tuple(range(1, n+1))
        # solo intercambios adyacentes
        self.actions = [(i, i+1) for i in range(n-1)]
        self.state = None

    def reset(self, initial=None):
        if initial is not None:
            self.state = list(initial)
        else:
            self.state = list(range(1, self.n+1))
            np.random.shuffle(self.state)
        return tuple(self.state)

    def step(self, action):
        i, j = action
        self.state[i], self.state[j] = self.state[j], self.state[i]
        state = tuple(self.state)
        done = state == self.goal
        reward = 1 if done else 0
        return state, reward, done

    def state_key(self):
        return str(self.state)

## Optimista

In [15]:
class AgentOptimista():
    def __init__(self, alpha=0.5, optimistic_value=1.0):
        self.value_function = {}
        self.alpha = alpha
        self.optimistic_value = optimistic_value
        self.positions = []
        self.actions_taken = []

    def reset(self):
        self.positions = []
        self.actions_taken = []

    def get_value(self, state):
        key = str(state)
        # estado desconocido -> valor optimista
        if key not in self.value_function:
            self.value_function[key] = self.optimistic_value
        return self.value_function[key]

    def move(self, state, actions):
        best_value = -1000
        best_action = actions[0]
        current = list(state)
        for action in actions:
            i, j = action
            next_state = current.copy()
            next_state[i], next_state[j] = next_state[j], next_state[i]
            value = self.get_value(next_state)
            if value > best_value:
                best_value = value
                best_action = action
        return best_action

    def update(self, state):
        self.positions.append(str(state))

    def reward(self, reward):
        for p in reversed(self.positions):
            if p not in self.value_function:
                self.value_function[p] = self.optimistic_value
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

## Gradiente

In [16]:
class AgentGradiente():
    def __init__(self, alpha=0.1):
        self.preferences = {}
        self.alpha = alpha
        self.positions = []
        self.pi_history = []      # probabilidades en cada paso
        self.actions_taken = []
        self.recompensas = []

    def reset(self):
        self.positions = []
        self.pi_history = []
        self.actions_taken = []

    def get_preference(self, state):
        key = str(state)
        return self.preferences.get(key, 0.0)

    def softmax(self, H):
        H = np.array(H)
        exp_H = np.exp(H - H.max())   # estabilidad numérica
        return exp_H / exp_H.sum()

    def move(self, state, actions):
        current = list(state)
        H = []
        next_states = []
        for action in actions:
            i, j = action
            next_state = current.copy()
            next_state[i], next_state[j] = next_state[j], next_state[i]
            H.append(self.get_preference(next_state))
            next_states.append(next_state)

        pi = self.softmax(H)
        # elegir acción según probabilidades
        ix = np.random.choice(len(actions), p=pi)

        self.pi_history.append((pi, ix))
        return actions[ix]

    def update(self, state):
        self.positions.append(str(state))

    def reward(self, reward):
        self.recompensas.append(reward)
        recompensa_media = np.mean(self.recompensas)

        # positions[0] es el estado inicial sin acción, por eso arrancamos desde 1
        for ix, p in enumerate(self.positions[1:]):
            if p not in self.preferences:
                self.preferences[p] = 0.0
            pi, chosen = self.pi_history[ix]
            for j in range(len(pi)):
                if j == chosen:
                    self.preferences[p] += self.alpha * (reward - recompensa_media) * (1 - pi[j])
                else:
                    self.preferences[p] -= self.alpha * (reward - recompensa_media) * pi[j]

In [17]:
def entrenar(agent, episodios=200000, max_pasos=20, initial=None):
    env = SortingEnv(n=5)
    exitos = 0

    for ep in tqdm(range(1, episodios + 1)):
        # alternar entre estado fijo y estados aleatorios para generalizar
        if initial is not None and ep % 2 == 0:
            state = env.reset(initial=initial)
        else:
            state = env.reset()  # estado aleatorio

        agent.reset()
        agent.update(state)
        recompensa_final = 0

        for paso in range(max_pasos):
            inversiones_antes = contar_inversiones(list(state))
            action = agent.move(state, env.actions)
            next_state, reward, done = env.step(action)
            agent.update(next_state)
            state = next_state

            if done:
                recompensa_final = 1
                exitos += 1
                break
            else:
                inversiones_despues = contar_inversiones(list(state))
                if inversiones_despues < inversiones_antes:
                    recompensa_final += 0.1
                elif inversiones_despues > inversiones_antes:
                    recompensa_final -= 0.05  # penalizar menos para no bloquear exploración

        agent.reward(recompensa_final)

    print(f"Episodios exitosos: {exitos}/{episodios} ({100*exitos/episodios:.1f}%)")
    return agent

# entrenar con el caso específico (5,3,4,2,1)
agent_opt = AgentOptimista(alpha=0.5, optimistic_value=1.0)
entrenar(agent_opt, episodios=50000, initial=(5,3,4,2,1))

agent_grad = AgentGradiente(alpha=0.1)
entrenar(agent_grad, episodios=50000, initial=(5,3,4,2,1))

  0%|          | 0/50000 [00:00<?, ?it/s]

100%|██████████| 50000/50000 [00:07<00:00, 7071.89it/s]


Episodios exitosos: 371/50000 (0.7%)


100%|██████████| 50000/50000 [01:31<00:00, 543.61it/s]

Episodios exitosos: 3398/50000 (6.8%)


In [18]:
def jugar(agent, initial=(5,3,4,2,1)):
    env = SortingEnv(n=5)
    state = env.reset(initial=initial)
    print(f"Inicio: {state}")

    for paso in range(20):
        action = agent.move(state, env.actions)
        state, reward, done = env.step(action)
        i, j = action
        print(f"Paso {paso+1}: intercambio posiciones {i} y {j} → {state}")
        if done:
            print("¡Ordenado!")
            break
    else:
        print(f"No ordenó. Estado final: {state}")

In [19]:
jugar(agent_opt)
jugar(agent_grad)

Inicio: (5, 3, 4, 2, 1)
Paso 1: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 2: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 3: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 4: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 5: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 6: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 7: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 8: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 9: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 10: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 11: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 12: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 13: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 14: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 15: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 16: intercambio posiciones 0 y 1 → (5, 3, 4, 2, 1)
Paso 17: intercambio posiciones 0 y 1 → (3, 5, 4, 2, 1)
Paso 18: intercambio posiciones 0